# Qwen3.5-27B Colab Inference Baseline

이 노트북은 Google Colab A100 40GB 단일 GPU에서 `Qwen/Qwen3.5-27B`를 8bit 양자화로 불러와 `test.csv`를 추론하는 운영용 버전입니다.

- LoRA / fine-tuning 없음
- reasoning 허용, 최종 답안은 `{"answer":"a"}` 형태 JSON으로 강제
- 문항별 JSON 로그 저장
- `START_INDEX` / `SKIP_EXISTING` 기반 재시작 지원
- 여러 실행 결과를 마지막에 하나의 `submission.csv`로 병합 가능
        


# 환경 준비

아래 설치 셀을 실행한 뒤 Colab에서 반드시 한 번 `런타임 다시 시작`을 해주세요.

- 권장 런타임: `A100 GPU 40GB`
- 이 노트북은 로컬 테스트를 하지 않고 Colab 실행을 전제로 작성되었습니다.
- 8bit + 128k 상한은 시도 가능한 구성으로 잡아두되, 실제 VQA 입력은 짧으므로 long-context 상한은 운영 설정값에 가깝습니다.
        


In [ ]:
!pip -q install -U "transformers>=4.57.6" accelerate bitsandbytes pandas pillow
        


# 데이터 준비

Google Drive는 사용하지 않습니다. 데이터 ZIP을 Colab 런타임에 직접 업로드한 뒤 `DATA_ZIP_PATH`만 맞춰서 압축을 풉니다.

- 예시 경로: `/content/2026-ssafy-15-2-ai.zip`
- 이미 `/content` 아래에 압축이 풀려 있으면 unzip 셀은 건너뛰어도 됩니다.
    


In [ ]:
try:
    from google.colab import files
except ImportError:
    files = None

print("데이터 ZIP을 Colab 런타임에 직접 업로드하세요.")
print("업로드 후 다음 셀의 DATA_ZIP_PATH를 실제 경로로 수정하면 됩니다.")
print("원하면 아래 주석을 해제해서 노트북 안에서 업로드할 수 있습니다.")

# uploaded = files.upload()
    


In [ ]:
DATA_ZIP_PATH = "/content/2026-ssafy-15-2-ai.zip"
EXTRACT_ROOT = "/content"
UNZIP_DATA = True

import os

if UNZIP_DATA:
    if os.path.exists(DATA_ZIP_PATH):
        !unzip -q -o "$DATA_ZIP_PATH" -d "$EXTRACT_ROOT"
        print(f"Unzipped: {DATA_ZIP_PATH} -> {EXTRACT_ROOT}")
    else:
        print("ZIP 파일을 찾지 못했습니다. DATA_ZIP_PATH를 업로드한 실제 경로로 수정하세요.")
else:
    print("UNZIP_DATA=False 이므로 압축 해제를 건너뜁니다.")
    


# 라이브러리, 실행 설정, 경로

여기서 실행 범위, 저장 위치, 그리고 브라우저 자동 백업 다운로드 주기를 조정합니다.
    


In [ ]:
import gc
import json
import os
import re
import tarfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3_5ForConditionalGeneration

try:
    from google.colab import files
except ImportError:
    files = None

Image.MAX_IMAGE_PIXELS = None

MODEL_ID = "Qwen/Qwen3.5-27B"
IMAGE_SIZE = 384
CONTEXT_LIMIT = 131072
MAX_NEW_TOKENS = 96
RETRY_MAX_NEW_TOKENS = 192
SAVE_EVERY = 100
SEED = 42

START_INDEX = 0
END_INDEX = None
RUN_NAME = "qwen35_27b_8bit_run01"
OUTPUT_ROOT = Path("/content/qwen35_runs")
SKIP_EXISTING = True

AUTO_DOWNLOAD_BACKUP = True
BACKUP_EVERY = 1
BACKUP_ROOT = Path("/content/qwen35_backups")

DATA_ROOT = Path("/content")
TEST_CSV_PATH = DATA_ROOT / "test.csv"

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RUN_DIR = OUTPUT_ROOT / RUN_NAME
ITEM_DIR = RUN_DIR / "items"
RUN_STATE_PATH = RUN_DIR / "run_state.json"
PROGRESS_PATH = RUN_DIR / "progress.csv"
MERGED_RESULTS_PATH = RUN_DIR / "merged_results.csv"
SUBMISSION_PATH = RUN_DIR / "submission.csv"
BACKUP_ARCHIVE_PATH = BACKUP_ROOT / f"{RUN_NAME}_checkpoint_backup.tar.gz"

for path in [OUTPUT_ROOT, RUN_DIR, ITEM_DIR, BACKUP_ROOT]:
    path.mkdir(parents=True, exist_ok=True)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def print_runtime_summary() -> None:
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        free_mem, total_mem = torch.cuda.mem_get_info()
        print("gpu:", name)
        print(f"free VRAM: {free_mem / 1024**3:.2f} GB / total VRAM: {total_mem / 1024**3:.2f} GB")
    print("MODEL_ID:", MODEL_ID)
    print("CONTEXT_LIMIT:", CONTEXT_LIMIT)
    print("RUN_DIR:", RUN_DIR)
    print("AUTO_DOWNLOAD_BACKUP:", AUTO_DOWNLOAD_BACKUP)
    print("BACKUP_EVERY:", BACKUP_EVERY)
    print("BACKUP_ARCHIVE_PATH:", BACKUP_ARCHIVE_PATH)


def resolve_image_path(relative_path: str) -> Path:
    path = DATA_ROOT / str(relative_path)
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")
    return path


def sanitize_file_token(text: str) -> str:
    return re.sub(r"[^0-9A-Za-z._-]+", "_", str(text))


def get_item_log_path(row_idx: int, sample_id: str) -> Path:
    safe_id = sanitize_file_token(sample_id)
    return ITEM_DIR / f"{row_idx:05d}_{safe_id}.json"


def write_json_atomic(path: Path, payload: Dict) -> None:
    temp_path = path.with_suffix(path.suffix + ".tmp")
    with temp_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    temp_path.replace(path)


def read_json(path: Path) -> Dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


print_runtime_summary()
    


# 모델, Processor 로드

- 8bit 양자화 로드
- `CONTEXT_LIMIT=128k`는 운영 상한선으로 사용
- 실제 입력이 짧기 때문에 기본 캐시는 동적 캐시를 유지하고, OOM 시에만 offloaded cache로 재시도합니다.
        


In [ ]:
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.eval()

tokenizer = processor.tokenizer
tokenizer.model_max_length = CONTEXT_LIMIT

print("processor:", processor.__class__.__name__)
print("tokenizer max length:", tokenizer.model_max_length)
if hasattr(model.config, "text_config") and hasattr(model.config.text_config, "max_position_embeddings"):
    print("model native max positions:", model.config.text_config.max_position_embeddings)
        


# 프롬프트, 파서, 추론 헬퍼

reasoning은 허용하되 최종 출력은 오직 JSON 한 개만 남기도록 강제합니다.
        


In [ ]:
SYSTEM_PROMPT = (
    "You are a careful multimodal reasoning assistant. "
    "Think step by step about the image and the multiple-choice options. "
    "After reasoning, output only valid JSON in exactly this form: "
    '{"answer":"a"} '
    "The answer value must be one lowercase letter among a, b, c, or d. "
    "Do not include markdown fences or any extra text outside the final JSON."
)

ANSWER_JSON_RE = re.compile(r'"answer"\s*:\s*"([abcd])"', re.IGNORECASE)
ANSWER_LETTER_RE = re.compile(r"\b([abcd])\b", re.IGNORECASE)


def build_mc_prompt(question: str, a: str, b: str, c: str, d: str) -> str:
    return (
        f"{question}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "Solve the question using the image. "
        "You may reason internally, but your final output must be only one JSON object like "
        '{"answer":"a"}.'
    )


def extract_post_think(text: str) -> str:
    text = text.strip()
    if "</think>" in text:
        return text.rsplit("</think>", 1)[-1].strip()
    return text


def truncate_text_for_context(text: str, token_budget: int) -> Tuple[str, bool]:
    token_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    if len(token_ids) <= token_budget:
        return text, False
    kept_ids = token_ids[:max(1, token_budget)]
    trimmed = tokenizer.decode(kept_ids, skip_special_tokens=True)
    return trimmed, True


def build_inputs(row: pd.Series) -> Tuple[Dict[str, torch.Tensor], Dict[str, object]]:
    image = Image.open(resolve_image_path(row["path"])).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    user_text, truncated_text = truncate_text_for_context(user_text, token_budget=CONTEXT_LIMIT // 2)
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": user_text},
            ],
        },
    ]

    chat_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    inputs = processor(text=[chat_text], images=[image], return_tensors="pt")
    input_token_count = int(inputs["input_ids"].shape[-1])

    second_truncation = False
    if input_token_count > CONTEXT_LIMIT:
        overflow = input_token_count - CONTEXT_LIMIT + 64
        original_token_ids = tokenizer(user_text, add_special_tokens=False)["input_ids"]
        keep_token_count = max(128, len(original_token_ids) - overflow)
        shortened_text = tokenizer.decode(original_token_ids[:keep_token_count], skip_special_tokens=True)
        messages[1]["content"][1]["text"] = shortened_text
        chat_text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
        inputs = processor(text=[chat_text], images=[image], return_tensors="pt")
        input_token_count = int(inputs["input_ids"].shape[-1])
        second_truncation = True

    if input_token_count > CONTEXT_LIMIT:
        raise RuntimeError(
            f"Input token count {input_token_count} exceeds CONTEXT_LIMIT={CONTEXT_LIMIT} even after truncation."
        )

    device_inputs = {}
    for key, value in inputs.items():
        if torch.is_tensor(value):
            device_inputs[key] = value.to(model.device)
        else:
            device_inputs[key] = value

    meta = {
        "input_token_count": input_token_count,
        "truncated_text": truncated_text or second_truncation,
        "chat_text_preview": chat_text[:400],
    }
    return device_inputs, meta


def parse_final_answer(text: str) -> Tuple[str, str, str]:
    full_text = text.strip()
    post_think = extract_post_think(full_text)
    candidates = [
        ("json_post_think", post_think),
        ("json_full_text", full_text),
    ]

    for method, candidate in candidates:
        match = ANSWER_JSON_RE.search(candidate)
        if match:
            return match.group(1).lower(), method, candidate[:2000]

        json_like_matches = re.findall(r"\{.*?\}", candidate, flags=re.DOTALL)
        for blob in reversed(json_like_matches):
            try:
                parsed = json.loads(blob)
            except json.JSONDecodeError:
                continue
            answer = str(parsed.get("answer", "")).strip().lower()
            if answer in {"a", "b", "c", "d"}:
                return answer, method + "_loads", blob[:2000]

    fallback_segments = [post_think, full_text]
    for idx, segment in enumerate(fallback_segments, start=1):
        lines = [line.strip() for line in segment.splitlines() if line.strip()]
        if lines:
            last_line = lines[-1]
            if last_line.lower() in {"a", "b", "c", "d"}:
                return last_line.lower(), f"line_fallback_{idx}", last_line[:2000]

            match = ANSWER_LETTER_RE.search(last_line)
            if match:
                return match.group(1).lower(), f"regex_last_line_{idx}", last_line[:2000]

        match = ANSWER_LETTER_RE.search(segment)
        if match:
            return match.group(1).lower(), f"regex_segment_{idx}", segment[:2000]

    return "a", "default_fallback", post_think[:2000]


def generate_with_retry(inputs: Dict[str, torch.Tensor]) -> Tuple[str, bool]:
    input_length = int(inputs["input_ids"].shape[-1])

    def _decode(output_ids: torch.Tensor) -> str:
        generated_ids = output_ids[:, input_length:]
        return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    try:
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
            )
        return _decode(output_ids), False
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gc.collect()
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=RETRY_MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                cache_implementation="offloaded",
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
            )
        return _decode(output_ids), True
        


# 실행 상태, 문항별 로그, 진행률 저장, 브라우저 백업 다운로드

문항 하나가 끝날 때마다 JSON으로 저장하고, 필요하면 현재 실행 디렉터리를 `tar.gz`로 묶어 브라우저 다운로드를 트리거합니다.
    


In [ ]:
RESULT_COLUMNS = [
    "row_idx",
    "id",
    "path",
    "answer",
    "status",
    "raw_final",
    "parse_method",
    "error",
    "input_token_count",
    "used_offloaded_cache",
    "truncated_text",
    "timestamp",
]


def load_run_state() -> Dict:
    if RUN_STATE_PATH.exists():
        state = read_json(RUN_STATE_PATH)
    else:
        state = {
            "run_name": RUN_NAME,
            "model_id": MODEL_ID,
            "created_at": utc_now(),
            "last_completed_row_idx": None,
            "last_completed_id": None,
            "processed_count": 0,
            "success_count": 0,
            "error_count": 0,
            "fallback_count": 0,
            "skipped_existing_count": 0,
            "updated_at": utc_now(),
        }
    return state


run_state = load_run_state()
session_stats = {
    "processed_this_session": 0,
    "success_this_session": 0,
    "error_this_session": 0,
    "skipped_existing_this_session": 0,
    "backup_downloads_this_session": 0,
}


def update_run_state(record: Dict, skipped_existing: bool = False) -> None:
    if skipped_existing:
        run_state["skipped_existing_count"] = int(run_state.get("skipped_existing_count", 0)) + 1
        run_state["updated_at"] = utc_now()
        return

    run_state["last_completed_row_idx"] = int(record["row_idx"])
    run_state["last_completed_id"] = record["id"]
    run_state["processed_count"] = int(run_state.get("processed_count", 0)) + 1
    run_state["updated_at"] = utc_now()

    if record["status"] == "ok":
        run_state["success_count"] = int(run_state.get("success_count", 0)) + 1
    elif record["status"] == "fallback":
        run_state["fallback_count"] = int(run_state.get("fallback_count", 0)) + 1
    else:
        run_state["error_count"] = int(run_state.get("error_count", 0)) + 1

    write_json_atomic(RUN_STATE_PATH, run_state)


def flush_run_state() -> None:
    run_state["updated_at"] = utc_now()
    write_json_atomic(RUN_STATE_PATH, run_state)


def create_backup_archive(
    run_names: Optional[List[str]] = None,
    archive_path: Optional[Path] = None,
) -> Path:
    run_names = run_names or [RUN_NAME]
    if archive_path is None:
        if len(run_names) == 1:
            archive_name = f"{run_names[0]}_checkpoint_backup.tar.gz"
        else:
            archive_name = f"merge__{'__'.join(run_names)}_checkpoint_backup.tar.gz"
        archive_path = BACKUP_ROOT / archive_name

    archive_path.parent.mkdir(parents=True, exist_ok=True)
    if archive_path.exists():
        archive_path.unlink()

    with tarfile.open(archive_path, "w:gz") as tar:
        for run_name in run_names:
            run_dir = OUTPUT_ROOT / run_name
            if run_dir.exists():
                tar.add(run_dir, arcname=f"qwen35_runs/{run_name}")

    return archive_path


def download_backup_archive(
    run_names: Optional[List[str]] = None,
    archive_path: Optional[Path] = None,
) -> Path:
    archive_path = create_backup_archive(run_names=run_names, archive_path=archive_path)
    print("Backup archive ready:", archive_path)
    if files is None:
        print("google.colab.files 를 사용할 수 없는 환경입니다.")
        return archive_path

    files.download(str(archive_path))
    session_stats["backup_downloads_this_session"] += 1
    return archive_path


def maybe_auto_download_backup(force: bool = False) -> Optional[Path]:
    if not AUTO_DOWNLOAD_BACKUP and not force:
        return None

    processed_count = int(run_state.get("processed_count", 0))
    if not force:
        if BACKUP_EVERY <= 0:
            return None
        if processed_count == 0 or processed_count % BACKUP_EVERY != 0:
            return None

    try:
        flush_run_state()
        return download_backup_archive(run_names=[RUN_NAME], archive_path=BACKUP_ARCHIVE_PATH)
    except Exception as exc:
        print("Backup download failed:", repr(exc))
        return None


def collect_item_logs(run_names: Optional[List[str]] = None) -> Tuple[pd.DataFrame, int]:
    run_names = run_names or [RUN_NAME]
    records: List[Dict] = []
    raw_count = 0

    for run_name in run_names:
        item_dir = OUTPUT_ROOT / run_name / "items"
        for path in sorted(item_dir.glob("*.json")):
            raw_count += 1
            record = read_json(path)
            record["run_name"] = run_name
            record["item_file"] = str(path)
            records.append(record)

    if not records:
        return pd.DataFrame(columns=RESULT_COLUMNS + ["run_name", "item_file"]), 0

    df = pd.DataFrame(records)
    df = df.sort_values(["row_idx", "timestamp", "run_name"]).drop_duplicates(
        subset=["row_idx", "id"], keep="last"
    )
    duplicate_count = raw_count - len(df)
    return df, duplicate_count


def refresh_progress_csv() -> pd.DataFrame:
    df, duplicate_count = collect_item_logs([RUN_NAME])
    if not df.empty:
        progress_df = df[
            [
                "row_idx",
                "id",
                "answer",
                "status",
                "parse_method",
                "error",
                "timestamp",
            ]
        ].sort_values("row_idx")
    else:
        progress_df = pd.DataFrame(columns=["row_idx", "id", "answer", "status", "parse_method", "error", "timestamp"])

    progress_df.to_csv(PROGRESS_PATH, index=False)
    flush_run_state()
    print(f"Progress CSV saved: {PROGRESS_PATH}")
    print("duplicate item logs within this run:", duplicate_count)
    return progress_df


refresh_progress_csv()
    


# 추론 실행

- `START_INDEX`, `END_INDEX`, `SKIP_EXISTING`, `AUTO_DOWNLOAD_BACKUP`, `BACKUP_EVERY`를 먼저 확인하세요.
- 각 문항 결과는 즉시 `items/*.json`으로 저장됩니다.
- `AUTO_DOWNLOAD_BACKUP=True`이면 저장 주기마다 현재 run 폴더를 `.tar.gz`로 묶어서 브라우저 다운로드를 시도합니다.
    


In [ ]:
test_df = pd.read_csv(TEST_CSV_PATH)
total_rows = len(test_df)

effective_end = total_rows if END_INDEX is None else min(int(END_INDEX), total_rows)
if START_INDEX < 0 or START_INDEX >= total_rows:
    raise ValueError(f"START_INDEX must be in [0, {total_rows - 1}]")
if effective_end <= START_INDEX:
    raise ValueError("END_INDEX must be greater than START_INDEX")

print("total test rows:", total_rows)
print("run range:", START_INDEX, "to", effective_end - 1)
print("skip existing:", SKIP_EXISTING)
print("auto backup download:", AUTO_DOWNLOAD_BACKUP)
print("backup every:", BACKUP_EVERY)

for row_idx in range(START_INDEX, effective_end):
    row = test_df.iloc[row_idx]
    item_path = get_item_log_path(row_idx, row["id"])

    if SKIP_EXISTING and item_path.exists():
        session_stats["skipped_existing_this_session"] += 1
        update_run_state({}, skipped_existing=True)
        if (row_idx - START_INDEX + 1) % SAVE_EVERY == 0:
            refresh_progress_csv()
        continue

    record = {
        "row_idx": int(row_idx),
        "id": str(row["id"]),
        "path": str(row["path"]),
        "answer": "a",
        "status": "error",
        "raw_final": "",
        "parse_method": "not_started",
        "error": "",
        "input_token_count": None,
        "used_offloaded_cache": False,
        "truncated_text": False,
        "timestamp": utc_now(),
    }

    try:
        inputs, meta = build_inputs(row)
        output_text, used_offloaded_cache = generate_with_retry(inputs)
        answer, parse_method, raw_final = parse_final_answer(output_text)

        status = "ok"
        if parse_method in {"default_fallback"}:
            status = "fallback"

        record.update(
            {
                "answer": answer,
                "status": status,
                "raw_final": raw_final,
                "parse_method": parse_method,
                "error": "",
                "input_token_count": meta["input_token_count"],
                "used_offloaded_cache": used_offloaded_cache,
                "truncated_text": meta["truncated_text"],
                "timestamp": utc_now(),
            }
        )

        session_stats["processed_this_session"] += 1
        if status == "ok":
            session_stats["success_this_session"] += 1
        else:
            session_stats["error_this_session"] += 1

    except Exception as exc:
        record.update(
            {
                "answer": "a",
                "status": "error",
                "raw_final": "",
                "parse_method": "exception_default",
                "error": repr(exc),
                "timestamp": utc_now(),
            }
        )
        session_stats["processed_this_session"] += 1
        session_stats["error_this_session"] += 1

    write_json_atomic(item_path, record)
    update_run_state(record)
    maybe_auto_download_backup()

    if (row_idx - START_INDEX + 1) % SAVE_EVERY == 0:
        print(
            f"[progress] row_idx={row_idx} processed={session_stats['processed_this_session']} "
            f"success={session_stats['success_this_session']} error_or_fallback={session_stats['error_this_session']} "
            f"skipped_existing={session_stats['skipped_existing_this_session']} "
            f"backup_downloads={session_stats['backup_downloads_this_session']}"
        )
        refresh_progress_csv()

refresh_progress_csv()
maybe_auto_download_backup(force=True)
flush_run_state()
print("run_state:", json.dumps(run_state, ensure_ascii=False, indent=2))
print("session_stats:", json.dumps(session_stats, ensure_ascii=False, indent=2))
    


# 병합 유틸리티

문항별 JSON 로그를 하나의 `merged_results.csv`와 제출용 `submission.csv`로 합칩니다.

- 단일 실행 병합: `MERGE_RUN_NAMES = [RUN_NAME]`
- 여러 실행 병합: `MERGE_RUN_NAMES = ["run01", "run02", "run03"]`
        


In [ ]:
def get_merge_output_dir(run_names: List[str], output_name: Optional[str] = None) -> Path:
    if output_name:
        out_dir = OUTPUT_ROOT / output_name
    elif len(run_names) == 1:
        out_dir = OUTPUT_ROOT / run_names[0]
    else:
        joined = "__".join(run_names)
        out_dir = OUTPUT_ROOT / f"merge__{joined}"
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir


def merge_run_outputs(
    run_names: Optional[List[str]] = None,
    output_name: Optional[str] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    run_names = run_names or [RUN_NAME]
    logs_df, duplicate_count = collect_item_logs(run_names)
    if logs_df.empty:
        raise ValueError(f"No item logs found for runs: {run_names}")

    logs_df = logs_df.sort_values(["row_idx", "timestamp", "run_name"]).drop_duplicates(
        subset=["row_idx"], keep="last"
    )

    merge_dir = get_merge_output_dir(run_names, output_name=output_name)
    merged_results_path = merge_dir / "merged_results.csv"
    submission_path = merge_dir / "submission.csv"

    base_df = pd.read_csv(TEST_CSV_PATH).reset_index().rename(columns={"index": "row_idx"})
    merged_df = base_df.merge(
        logs_df[
            [
                "row_idx",
                "id",
                "answer",
                "status",
                "raw_final",
                "parse_method",
                "error",
                "input_token_count",
                "used_offloaded_cache",
                "truncated_text",
                "timestamp",
                "run_name",
            ]
        ],
        on=["row_idx", "id"],
        how="left",
    )

    missing_count = int(merged_df["answer"].isna().sum())
    invalid_answer_count = int((~merged_df["answer"].fillna("").isin(list("abcd"))).sum())

    submission_df = merged_df[["id", "answer"]].copy()
    submission_df["answer"] = submission_df["answer"].fillna("a")

    merged_df.to_csv(merged_results_path, index=False)
    submission_df.to_csv(submission_path, index=False)

    print("merge_dir:", merge_dir)
    print("duplicate item logs:", duplicate_count)
    print("missing answers:", missing_count)
    print("invalid answers:", invalid_answer_count)
    print("merged_results.csv:", merged_results_path)
    print("submission.csv:", submission_path)

    return merged_df, submission_df


MERGE_RUN_NAMES = [RUN_NAME]
# 예시:
# MERGE_RUN_NAMES = ["qwen35_27b_8bit_run01", "qwen35_27b_8bit_run02"]
        


In [ ]:
merged_results_df, submission_df = merge_run_outputs(run_names=MERGE_RUN_NAMES)

print(merged_results_df.head(3).to_string())
print(submission_df.head(3).to_string())
        


# 수동 백업 다운로드

자동 다운로드 외에도, 원하는 시점에 현재 run 폴더를 하나의 `tar.gz`로 묶어 직접 내려받을 수 있습니다.
    


In [ ]:
backup_path = create_backup_archive(run_names=[RUN_NAME], archive_path=BACKUP_ARCHIVE_PATH)
print("Backup archive ready:", backup_path)

if files is not None:
    files.download(str(backup_path))
else:
    print("google.colab.files 를 사용할 수 없는 환경입니다.")
    


# 선택 사항: 8bit + 128k 메모리 프로브

이 셀은 실제 제출 추론과 별개입니다. Colab에서 8bit + 긴 컨텍스트가 어느 정도까지 버티는지 보고 싶을 때만 수동으로 실행하세요.

- 기본 VQA 입력은 짧아서 이 테스트가 필수는 아닙니다.
- OOM이 나면 즉시 중단하고 `MAX_NEW_TOKENS` 또는 이미지 픽셀 예산을 먼저 줄이는 쪽을 권장합니다.
        


In [ ]:
def long_context_probe(token_targets: List[int] = [8000, 32000, 64000, 128000]) -> None:
    probe_image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), color="white")
    base_chunk = "Recycling rules and observations. " * 256

    for target_tokens in token_targets:
        print(f"\n[probe] target_tokens={target_tokens}")
        repeated_text = base_chunk
        while True:
            token_count = len(tokenizer(repeated_text, add_special_tokens=False)["input_ids"])
            if token_count >= target_tokens:
                break
            repeated_text += base_chunk

        prompt_text = (
            repeated_text
            + "\n\nQuestion: Which option is correct?\n(a) metal\n(b) glass\n(c) paper\n(d) plastic\n"
            + 'Return only {"answer":"a"}.'
        )

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": probe_image},
                    {"type": "text", "text": prompt_text},
                ],
            },
        ]

        chat_text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
        inputs = processor(text=[chat_text], images=[probe_image], return_tensors="pt")
        inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

        try:
            with torch.no_grad():
                _ = model.generate(
                    **inputs,
                    max_new_tokens=8,
                    do_sample=False,
                    use_cache=True,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.eos_token_id,
                )
            print("status: success")
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            gc.collect()
            print("status: OOM")
            break
        except Exception as exc:
            print("status: error", repr(exc))
            break
        


# Colab 실행 / 검증 방법

## 1. 데이터 ZIP 업로드
- Google Drive 대신 데이터 ZIP을 Colab 런타임에 직접 업로드합니다.
- 예시 경로는 `/content/2026-ssafy-15-2-ai.zip`입니다.
- 업로드 후 `DATA_ZIP_PATH`를 실제 경로로 맞추고 압축 해제 셀을 실행합니다.

## 2. 런타임 확인
- Colab 런타임이 `A100 GPU 40GB`인지 먼저 확인합니다.
- 모델 로드 전후로 VRAM 사용량을 확인합니다.

## 3. 설치 후 재시작
- 패키지 설치 셀 실행 후 Colab에서 `런타임 다시 시작`을 한 번 수행합니다.
- 재시작 후에는 위에서부터 순서대로 다시 실행합니다.

## 4. 3문항 스모크 테스트
- 설정 셀에서 `START_INDEX = 0`, `END_INDEX = 3`, `RUN_NAME = "smoke_run"`처럼 바꿉니다.
- 처음에는 `AUTO_DOWNLOAD_BACKUP = True`, `BACKUP_EVERY = 1` 또는 `2`로 두면 문항별 다운로드 동작을 바로 확인할 수 있습니다.
- 추론 실행 셀을 돌린 뒤 `/content/qwen35_runs/smoke_run/items/` 아래에 JSON 파일이 생기는지 확인합니다.

## 5. 실시간 백업 다운로드 확인
- 추론 셀 실행 중 저장 주기가 도달할 때마다 `/content/qwen35_backups/{RUN_NAME}_checkpoint_backup.tar.gz`가 갱신됩니다.
- `AUTO_DOWNLOAD_BACKUP=True`이면 `from google.colab import files; files.download(...)` 방식으로 브라우저 다운로드가 트리거됩니다.
- 브라우저 다운로드가 너무 자주 뜨면 `BACKUP_EVERY`를 `5`, `10`, `50`처럼 올리면 됩니다.

## 6. 재시작 테스트
- 같은 `RUN_NAME`을 유지한 채 일부만 실행한 후 세션을 끊거나 중단합니다.
- 다시 연결한 뒤 `START_INDEX = 0`, `SKIP_EXISTING = True`로 실행합니다.
- 이미 저장된 문항은 건너뛰고 이어서 진행되는지 확인합니다.
- 필요하면 수동 백업 다운로드 셀로 현재 상태를 즉시 내려받습니다.

## 7. 병합 테스트
- `MERGE_RUN_NAMES = [RUN_NAME]` 상태에서 병합 셀을 실행합니다.
- `merged_results.csv`와 `submission.csv`가 생성되는지 확인합니다.
- `submission.csv`의 행 수가 `test.csv`와 같은지 확인합니다.
- `answer` 컬럼 값이 모두 `a`, `b`, `c`, `d` 중 하나인지 확인합니다.

## 8. 여러 실행 병합 테스트
- 예를 들어 `run01`, `run02`, `run03`처럼 나눠 돌렸다면 `MERGE_RUN_NAMES = ["run01", "run02", "run03"]`로 바꿔 병합합니다.
- 병합 결과에서 누락 문항 수, 중복 로그 수, 잘못된 답안 수를 출력값으로 확인합니다.

## 9. 8bit + 128k 운영 가이드
- 현재 과제 입력은 짧은 VQA라서 `CONTEXT_LIMIT = 131072`는 상한 설정값에 가깝습니다.
- 그래도 세션 상태에 따라 OOM이 날 수 있으므로, 이 노트북은 OOM 발생 시 `offloaded` cache로 1회 재시도합니다.
- 그래도 불안정하면 `MAX_NEW_TOKENS`를 먼저 줄이고, 그 다음 `IMAGE_SIZE`를 줄이는 순서로 완화하는 것을 권장합니다.
- 긴 컨텍스트 자체를 실험하고 싶을 때만 `long_context_probe()`를 수동 실행하세요.
    
